# Session 2 · Day 2 - Assessor Workload and Coverage

**Assessor scoping** reads an assessor roster, their certificates, and their clock-in / clock-out time logs to answer three operational questions for TESDA: who is overloaded, who is idle, and which centers lean on too few people. The work here is tabular data cleaning plus a handful of capacity KPIs. *Routing and modeling come later in the session.*

> ⚠️ **The assessor data in this notebook is SYNTHETIC / illustrative.** It pairs realistic made-up assessors and time logs with real NCR assessment centers and their real accredited qualifications. The methods are real; the findings describe invented people and must not be reported as facts about TESDA.

### What this notebook covers
1. Load three tables and drop personal columns on load
2. Clean the time logs for four problems, counting and deciding each
3. Turn logs into hours and roll them up by week
4. Measure utilization against an assumed available-hours denominator
5. Separate active from inactive assessors on a configurable recency window
6. Rank center load: candidates per own assessor, and borrowed share
7. List certified assessors whose assigned center sits within 5 km of the busiest center

> **How to use this notebook**
> - ▶ **Run each cell in order** (`Shift+Enter`). Later cells depend on earlier ones.
> - ✏️ **Exercise cells** carry `### START CODE HERE ###` … `### END CODE HERE ###`. Edit only between those markers.
> - ✅ After an exercise, run the **check cell**. Fix the code until it prints a pass.
> - 🐞 **Spot-the-flaw cells** show a plausible but wrong claim. Find the reason before reading the note.
> - 🎛️ **Play-around cells** are open to tweak: change a number, re-run, compare.
>
> _Data loads from a public bucket over the network. Look for **`# >>> SWAP IN REAL DATA`** to point at a different source._

### New to Python or pandas?

Anyone coming from another language can run `S2_commands_primer` first. It lays out the mental model in one page: a `DataFrame` is a table (a SQL table, an R `data.frame`, a spreadsheet), a `merge` is a `JOIN`, `NaN` is the missing value, and a chain of `.method().method()` reads left to right. The comments in the code cells point out each pandas idiom as it appears.

In [ ]:
# Run once per Colab session (skip if already installed locally).
!pip -q install pandas pyarrow matplotlib numpy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 55)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# >>> SWAP IN REAL DATA: point BASE at a different bucket or local folder to change source.
BASE = "https://storage.googleapis.com/tesda-datasets/data/"

## Three tables, personal data dropped on load

| File | One row per… |
|---|---|
| `assessors.csv` | assessor: home barangay and city, primary center, employment type |
| `assessor_qualifications.csv` | one assessor × qualification **certificate**, with its validity dates |
| `assessor_timelogs.csv` | one time-in / time-out entry: who, where, what activity, which qualification, how many candidates |

`assessors.csv` also carries `full_name` and `mobile_no`. Neither is needed: an `assessor_id` is enough to count and link. The loader keeps only the useful columns with `usecols`, so the personal columns never enter memory. The data is synthetic, but the habit is the point.

In [ ]:
assessors = pd.read_csv(
    BASE + "assessors.csv",
    usecols=["assessor_id", "home_barangay", "home_city",
             "primary_center_id", "employment_type"],
)   # full_name and mobile_no are personal data: usecols drops them on load
certificates = pd.read_csv(
    BASE + "assessor_qualifications.csv",
    usecols=["assessor_id", "qualification", "certified_on", "valid_until"],
    parse_dates=["certified_on", "valid_until"],   # read these text columns straight into datetime
)
logs_raw = pd.read_csv(
    BASE + "assessor_timelogs.csv",
    parse_dates=["time_in", "time_out"],
)
print(f"assessors: {len(assessors):,}  certificates: {len(certificates):,}  log entries: {len(logs_raw):,}")

# The period the logs cover. "Today" for this notebook is the last logged day.
START = logs_raw["time_in"].min().normalize()   # normalize() drops the time-of-day, leaving midnight
END = logs_raw["time_in"].max().normalize()
N_WEEKS = ((END - START).days + 1) / 7
print(f"Period: {START:%d %b %Y} to {END:%d %b %Y}  ({N_WEEKS:.1f} weeks)")

One row of `assessors` is one assessor. One row of `certificates` is one certificate the assessor holds for one qualification. One row of `logs_raw` is one clock-in / clock-out entry. An assessor can have several entries in a day: `Travel` to the center, an `Orientation`, the `Assessment` itself (with `candidates_assessed`), then `Documentation`. The `qualification` field is blank for travel. Peek at the roster, then count the activity types.

In [ ]:
assessors.head(3)

In [ ]:
logs_raw["activity"].value_counts()   # value_counts = GROUP BY activity + COUNT, ordered by count

## Clean the time logs

Logs typed by people carry four predictable problems. Count each, decide what to do, and check the row count so nothing changes silently.

| # | Problem | Detection | Decision |
|---|---|---|---|
| A | no time-out (forgot to clock out) | `time_out` is blank | drop from hours, flag to the assessor |
| B | time-out at or before time-in | `time_out <= time_in` | drop (cannot tell which time is wrong) |
| C | the same entry submitted twice | identical in every column except `log_id` | keep one |
| D | work logged on an expired certificate | assessment after the certificate's `valid_until` | keep the hours, flag for compliance |

Problem A is the missing time-out. `isna()` is `True` where a value is blank.

In [ ]:
missing_out = logs_raw["time_out"].isna()   # boolean mask: True where the value is blank (NULL)
print(f"A - entries with no time-out: {missing_out.sum():,} ({missing_out.mean():.1%})")   # mean of a boolean mask = fraction True

Problem B is a time-out at or before the time-in.

In [ ]:
reversed_times = logs_raw["time_out"] <= logs_raw["time_in"]   # a comparison on a column returns a boolean mask
print(f"B - entries whose time-out is not after the time-in: {reversed_times.sum():,}")
logs_raw.loc[reversed_times, ["log_id", "assessor_id", "activity", "time_in", "time_out"]].head(3)   # .loc[mask, cols]: rows where mask is True, these columns

Problem C is the duplicate. `duplicated(subset=...)` marks every repeat after the first. The comparison uses every column except `log_id`, because a double-submitted form gets a fresh ID each time.

In [ ]:
compare_cols = [c for c in logs_raw.columns if c != "log_id"]   # list comprehension: every column name except log_id
duplicate = logs_raw.duplicated(subset=compare_cols, keep="first")
print(f"C - duplicate entries: {duplicate.sum():,}")

Apply A, B, and C together, then confirm the row count with an `assert` so nothing is removed silently.

In [ ]:
logs = logs_raw[~missing_out & ~reversed_times & ~duplicate].copy()   # ~ is NOT, & is AND on masks (not Python's not/and)
removed = len(logs_raw) - len(logs)
print(f"rows before: {len(logs_raw):,}  removed: {removed:,}  rows after: {len(logs):,}")

assert len(logs) == len(logs_raw) - (missing_out | reversed_times | duplicate).sum()
assert not logs["time_out"].isna().any()
assert (logs["time_out"] > logs["time_in"]).all()

flagged_missing_out = logs_raw.loc[missing_out, ["log_id", "assessor_id", "center_id", "activity", "time_in"]]
print(f"flagged for follow-up (missing time-out): {len(flagged_missing_out):,}")

Problem D is work logged on an expired certificate. Matching it needs a join, each assessment paired to the certificate the assessor holds for that qualification. `how="left"` keeps every assessment even when no certificate matches, and the row-count `assert` confirms the merge did not duplicate rows.

In [ ]:
assessments = logs[logs["activity"] == "Assessment"]
with_cert = assessments.merge(   # merge = SQL JOIN, here on the (assessor_id, qualification) pair
    certificates[["assessor_id", "qualification", "valid_until"]],
    on=["assessor_id", "qualification"], how="left", validate="many_to_one",
)
assert len(with_cert) == len(assessments)

no_cert = with_cert["valid_until"].isna()   # left join left the unmatched assessments as NaN here
expired = with_cert["time_in"] > with_cert["valid_until"]
print(f"D - assessments with no matching certificate: {no_cert.sum():,}")
print(f"D - assessments logged after the certificate expired: {expired.sum():,} "
      f"(by {with_cert.loc[expired, 'assessor_id'].nunique()} assessors)")   # nunique = COUNT DISTINCT

No assessments appear without a matching certificate, but several hundred are logged after a certificate expired. Each of those is a compliance finding: the reader for that list is TESDA's certification office.

## From logs to hours

Subtracting two timestamps gives a duration. `.dt.total_seconds() / 3600` turns it into hours. A quick sanity bound: a single assessment entry does not run 14 hours, so the longest entries per activity are worth a look.

In [ ]:
logs["hours"] = (logs["time_out"] - logs["time_in"]).dt.total_seconds() / 3600   # .dt reaches datetime ops on a column; subtracting timestamps gives a duration
print(logs["hours"].describe().round(2))   # describe() = summary stats, like R's summary()
logs.groupby("activity")["hours"].agg(["count", "median", "max"]).round(2)   # groupby = SQL GROUP BY; agg lists the aggregates

Now the weekly picture per assessor. `dt.to_period("W")` labels each entry with its week, and the sum per assessor-week shows the spread of weekly load.

In [ ]:
logs["week"] = logs["time_in"].dt.to_period("W")   # label each timestamp with its calendar week
weekly = logs.groupby(["assessor_id", "week"])["hours"].sum().reset_index()   # reset_index: turn the group keys back into plain columns
weekly["hours"].describe().round(1)

## Utilization as hours worked over hours available

Utilization is the headline capacity KPI. It needs a denominator: how many hours an assessor could work. The roster records `employment_type` but no contracted hours, so the denominator is an explicit assumption, stated here:

| employment type | assumed hours available per week |
|---|---|
| Full-time | 40 |
| Part-time | 20 |
| Freelance | 16 |

$$\text{utilization} = \frac{\text{hours logged in the period} \;/\; \text{weeks in the period}}{\text{assumed hours available per week}}$$

An assessor at 1.0 works exactly the assumed hours. Above 1.0 is overloaded, near 0 is barely used. The denominator is a modeling choice, so the utilization numbers are only as trustworthy as that assumption.

In [ ]:
AVAILABLE_HOURS = {"Full-time": 40, "Part-time": 20, "Freelance": 16}

hours_total = logs.groupby("assessor_id")["hours"].sum()
workload = assessors.set_index("assessor_id")[["primary_center_id", "employment_type", "home_city"]].copy()   # set_index: key the rows by assessor_id
workload["hours_per_week"] = (hours_total / N_WEEKS).reindex(workload.index).fillna(0)   # reindex aligns to workload's rows; fillna(0) = assessors with no logs
workload["available_per_week"] = workload["employment_type"].map(AVAILABLE_HOURS)   # .map(dict) = per-value lookup, like a VLOOKUP
workload.head(3)

## ✏️ Exercise - compute utilization

`workload` already holds `hours_per_week` and `available_per_week`. Fill in the `utilization` column: hours per week divided by available hours per week. Store it in `workload["utilization"]`.

In [ ]:
### START CODE HERE ###   (about 1 line)
workload["utilization"] = None   # hours_per_week divided by available_per_week
### END CODE HERE ###
workload["utilization"].describe().round(2)

## ✅ Check the work
Run this. It recomputes the reference and compares it to the computed column.

In [ ]:
ref_util = workload["hours_per_week"] / workload["available_per_week"]
try:
    assert workload["utilization"].notna().any(), "utilization column not filled in"
    assert np.allclose(workload["utilization"].values, ref_util.values, equal_nan=True), \
        "values do not match hours_per_week / available_per_week"
    print(f"✅ Passed! Median utilization {workload['utilization'].median():.0%}, "
          f"max {workload['utilization'].max():.0%}.")
except AssertionError as e:
    print("❌ Not yet:", e)

A histogram reads the spread at a glance. Values above 2.5 are clipped to 2.5 so the long tail does not flatten the rest.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(workload["utilization"].clip(upper=2.5), bins=np.arange(0, 2.6, 0.1),
        color="#30638e", edgecolor="white")
ax.axvline(1, color="#d1495b", linestyle="--")
ax.text(1.03, ax.get_ylim()[1] * 0.9, "100% of available hours", color="#d1495b", fontsize=8)
ax.set_xlabel("utilization (values above 2.5 drawn at 2.5)")
ax.set_ylabel("assessors")
ax.set_title(f"{len(workload)} assessors - median utilization {workload['utilization'].median():.0%}")
plt.show()
print(f"Overloaded (>100%): {(workload['utilization'] > 1).sum()}  "
      f"barely used (<10%): {(workload['utilization'] < 0.1).sum()}")

Most assessors sit well under their available hours, and a few dozen run far over, some at two or three times. Both ends matter: the overloaded are a quality and burnout risk, the barely-used are certified capacity that is not being deployed.

### ❓ Question

The utilization chart marks assessors above 100% of their available hours. In `workload`, how many assessors run over 100% utilization (`utilization > 1`)? Fill in the integer.

In [ ]:
q_overloaded = None   # fill in the integer

In [ ]:
_ref = int((workload["utilization"] > 1).sum())
if q_overloaded == _ref:
    print(f"✅ Correct! {_ref} assessors run over 100% utilization.")
else:
    print(f"❌ Not yet: expected {_ref}, got {q_overloaded}.")

## 🐞 Spot the flaw

A draft report reads: *"Assessor A0055 runs at 383% utilization, the hardest-working assessor in NCR. Route more candidates here."*

Run the cell. The number is honest arithmetic on the logs, but the conclusion is wrong. Find the reason before reading the note.

In [ ]:
busiest_assessor = workload["utilization"].idxmax()   # idxmax returns the index label of the max, not the value itself
workload.loc[[busiest_assessor],
             ["employment_type", "hours_per_week", "available_per_week", "utilization"]].round(2)

> The flagged 383% is about 77 logged hours a week against an assumed 20 for a part-timer. No person sustains that across six months, so the honest reading points to a data problem: a stale `employment_type` (this assessor may be full-time now), two people sharing one ID, or logs attributed to the wrong assessor. A utilization far above 100% is a signal to verify.

## Active vs inactive assessors

The second capacity KPI is the active-to-inactive ratio, and it needs a rule. The rule here: an assessor is **active** when the last logged entry falls within `ACTIVE_WINDOW_DAYS` of the end of the period. Assessors with no logs at all read as **never logged**.

In [ ]:
ACTIVE_WINDOW_DAYS = 60
last_log = logs.groupby("assessor_id")["time_in"].max()
workload["last_log"] = last_log.reindex(workload.index)   # align to workload's rows; no-log assessors become NaT
workload["status"] = np.select(   # np.select = vectorized if/elif/else: first matching condition wins
    [workload["last_log"].isna(),
     workload["last_log"] < END - pd.Timedelta(days=ACTIVE_WINDOW_DAYS)],
    ["never logged", "inactive"], default="active")

counts = workload["status"].value_counts()
active = counts.get("active", 0)   # .get with a default, like a dict lookup
dormant = counts.get("inactive", 0) + counts.get("never logged", 0)
print(counts.to_string())
print(f"Active-to-inactive ratio: {active / dormant:.1f} : 1")

Most assessors are active, and a meaningful minority are inactive or have never logged anything across six months. "Never logged" can mean a certified assessor who is never called, or one who left and was never removed. Either way, a plain count of certified assessors overstates the capacity actually on hand.

## 🎛️ Play around - change the active window

The active-to-inactive ratio depends entirely on the window. Change the days in the tuple below and watch the three counts move. A short window calls more part-timers inactive. A long one hides genuine dropout.

In [ ]:
for window_days in (30, 60, 90):
    cutoff = END - pd.Timedelta(days=window_days)
    status = np.select(
        [workload["last_log"].isna(), workload["last_log"] < cutoff],
        ["never logged", "inactive"], default="active")
    vc = pd.Series(status).value_counts()
    print(f"window {window_days:>3}d -> active {vc.get('active', 0):>3}, "
          f"inactive {vc.get('inactive', 0):>3}, never logged {vc.get('never logged', 0):>3}")

## Which centers carry the candidates

Group by center instead of assessor. For each center: `candidates` assessed there, `own_assessors` whose primary center it is, and `borrowed_share`, the share of its assessments done by assessors based elsewhere. A center with many candidates per own assessor and a high borrowed share is leaning on other centers' people.

In [ ]:
assessments = logs[logs["activity"] == "Assessment"].merge(   # merge = SQL JOIN to attach each assessor's home center
    assessors[["assessor_id", "primary_center_id"]], on="assessor_id", how="left")
assessments["borrowed"] = assessments["center_id"] != assessments["primary_center_id"]   # worked somewhere other than the home center

centers = assessments.groupby("center_id").agg(   # named aggregation: new_col=(source_col, function)
    candidates=("candidates_assessed", "sum"),
    assessment_entries=("log_id", "size"),
    borrowed_share=("borrowed", "mean"))   # mean of the boolean = share that were borrowed
centers["own_assessors"] = (assessors.groupby("primary_center_id").size()
                            .reindex(centers.index).fillna(0).astype(int))
centers["candidates_per_own_assessor"] = centers["candidates"] / centers["own_assessors"].replace(0, np.nan)   # 0 -> NaN so the division yields NaN, not a divide-by-zero

coords = pd.read_parquet(BASE + "tesda_coordinates.parquet")
centers = centers.join(   # join = merge on the index (tesda_inst_id)
    coords.set_index("tesda_inst_id")[["name", "latitude", "longitude"]], how="left")
centers.sort_values("candidates_per_own_assessor", ascending=False).head(10)[
    ["name", "candidates", "own_assessors", "candidates_per_own_assessor", "borrowed_share"]].round(2)

The top centers assessed well over a thousand candidates in six months with a single own assessor. That assessor is almost certainly among the overloaded from the utilization section. Confirm it for the busiest center:

In [ ]:
top_center = centers["candidates_per_own_assessor"].idxmax()   # index label (center_id) of the max, not the value
workload[workload["primary_center_id"] == top_center][   # boolean filter: assessors whose home center is the busiest one
    ["employment_type", "hours_per_week", "utilization", "status"]].round(2)

## Whose assigned center is within 5 km of the busiest center?

When one center leans on a single assessor, the next question is whether certified help is based nearby. "Nearby" here is **straight-line** distance from each assessor's assigned center (their office) to the busiest center, kept light so the notebook runs without a routing engine. Each assessor's office comes from matching their `primary_center_id` to the center coordinates loaded above.

> **Lecture note.** Straight-line distance is a lower bound on the real trip: the road always bends, so road distance is at least as long and usually longer. A straight-line 5 km filter therefore returns a *superset* of the assessors a true road-network query (the OSRM routing earlier in Session 2 Day 2) would keep. Treat this list as a first pass, to be narrowed by road distance.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    radius_km = 6371.0088
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * radius_km * np.arcsin(np.sqrt(a))   # vectorized: runs on single numbers or whole columns

CUTOFF_KM = 5
center_row = centers.loc[top_center]
qualification = (assessments.loc[assessments["center_id"] == top_center, "qualification"]
                 .value_counts().index[0])   # index[0] of value_counts = the most frequent value (the mode)

# Office = each assessor's assigned center. Match primary_center_id to the center coordinates.
office = (coords.set_index("tesda_inst_id")[["latitude", "longitude"]]
          .rename(columns={"latitude": "office_lat", "longitude": "office_lon"}))
based = assessors.merge(office, left_on="primary_center_id", right_index=True, how="left")   # right_index=True: join against the right frame's index
no_office = based["office_lat"].isna()
if no_office.any():
    print(f"note: {no_office.sum()} assessors have an assigned center with no coordinates "
          f"and are dropped from the proximity list")
based = based[~no_office]

valid_cert = certificates[(certificates["qualification"] == qualification) &
                          (certificates["valid_until"] >= END)]["assessor_id"]
nearby = based[based["assessor_id"].isin(valid_cert)].copy()   # .isin = SQL IN (membership test)
nearby["straight_km_to_center"] = haversine_km(
    nearby["office_lat"], nearby["office_lon"], center_row["latitude"], center_row["longitude"])
nearby = nearby[nearby["straight_km_to_center"] <= CUTOFF_KM].merge(
    workload[["status", "utilization"]], left_on="assessor_id", right_index=True)

assert len(nearby) <= valid_cert.nunique()
print(f"Center: {center_row['name']}")
print(f"Qualification: {qualification}")
print(f"Certified (valid) assessors in the dataset: {valid_cert.nunique()}  "
      f"with an assigned center within {CUTOFF_KM} km straight-line: {len(nearby)}")
nearby.sort_values("straight_km_to_center")[
    ["assessor_id", "primary_center_id", "home_city",
     "straight_km_to_center", "status", "utilization"]].round(2)

Each row is a possible helper. The useful ones are active and carry low utilization: an assessor already over 100% is not spare capacity, even next door. That is the scoping rule in miniature: near enough, certified, active, and not already overloaded.

## Recap
- Three tables, three grains: assessor, certificate, time-log entry. Personal columns dropped on load with `usecols`.
- Four cleaning problems (missing time-out, reversed times, duplicates, expired certificates), each counted, decided, and row-count checked. One is a compliance finding.
- Utilization needs a denominator the data does not hold. The assumption is stated, and a value far above 100% is a flag to verify.
- Active vs inactive depends on a chosen window, and a plain count of certified assessors overstates real capacity.
- A straight-line 5 km filter gives a first-pass shortlist of nearby certified help, to be narrowed later by road distance.

## For discussion
1. Which of these KPIs belong in a monthly management view, and which belong only with the certification office?
2. If assessors clocked in on an app, which of the four cleaning problems could the app prevent at entry?
3. The shortlist measures distance from each assessor's assigned center. When would "from home" be the better origin?

## References & further reading

- [pandas, user guide](https://pandas.pydata.org/docs/user_guide/)
- [pandas, group by (split-apply-combine)](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [pandas, merge and join](https://pandas.pydata.org/docs/user_guide/merging.html)
- [pandas, time series and date offsets](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [Haversine formula (great-circle distance)](https://en.wikipedia.org/wiki/Haversine_formula)
- [TESDA competency assessment and certification](https://www.tesda.gov.ph/)